In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torchmetrics.functional import structural_similarity_index_measure as ssim

In [2]:
face_transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root='Data', transform=face_transform)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

In [3]:
class BetaVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(BetaVAE, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1), # 32*24*24 
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),# 64*12*12
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),# 128*6*6
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),# 256*3*3
            nn.ReLU(),
            nn.Flatten() # 256*3*3 = 2304
        )
        
        # Latent Space: Linear layers for Mean and Log-Variance
        self.fc_mu = nn.Linear(2304, latent_dim)
        self.fc_var = nn.Linear(2304, latent_dim)
        
        # Decoder: Linear to expand latent dim, then ConvTranspose to 48x48
        self.decoder_input = nn.Linear(latent_dim, 2304)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), # 128*6*6
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), # 64*12*12
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # 32*24*24
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1), # 1*48*48
            nn.Sigmoid()
        )


    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        features = self.encoder(x)
        mu = self.fc_mu(features)
        logvar = self.fc_var(features)
        
        z = self.reparameterize(mu,logvar)
        
        x_hat = self.decoder_input(z)
        x_hat = x_hat.view(-1,256,3,3)
        reconstruction = self.decoder(x_hat)
        
        return reconstruction, mu, logvar

In [4]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    recon_loss = F.l1_loss(recon_x, x, reduction='sum')
    kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kld_loss

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BetaVAE(latent_dim=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ONE BATCH TEST
# x,_ = next(iter(train_loader))
# x = x.to(device)
# recon_x, mu, logvar = model(x)
# print(recon_x.shape)
# print(mu,logvar)

# loss = vae_loss(recon_x,x,mu,logvar,beta=2.0)
# print("loss is:",loss)
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()


# BATCH TEST
# for batch_idx, (x, _) in enumerate(train_loader):
#     x = x.to(device)
    
#     # 1. Forward Pass
#     recon_x, mu, logvar = model(x)
    
#     # 2. Calculate Loss
#     loss = vae_loss(recon_x, x, mu, logvar, beta=2.0)
    
#     # 3. Backprop
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

In [6]:
NUM_EPOCHS = 1

for epoch in range(NUM_EPOCHS):
    model.train() # Set to training mode
    train_loss = 0
    
    for batch_idx, (x, _) in enumerate(train_loader):
        x = x.to(device)

        # 1. Forward Pass
        recon_x, mu, logvar = model(x)

        # 2. Calculate Loss
        loss = vae_loss(recon_x, x, mu, logvar, beta=1.0)
        train_loss += loss.item()
        # 3. Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f'Epoch {epoch}: Average loss: {train_loss / len(train_loader.dataset):.4f}')

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'beta_vae_weights0.pth')